# RadioSkySims — Simulação de TOD (leve)

Pipeline de geração de Time-Ordered Data (TOD) do RadioSkySim **desacoplado**
(`radioskysims_simulation`), usando a API `SimOrchestrator` e o perfil
`uirapuru` do `settings.toml`.

Este notebook é uma **simulação leve**: janela de 1 h (passo 60 s), apenas GPS,
com `use_processor=True` para batches seguros de memória. Valida o ciclo
**satélites → TOD → persistência** sem travar a máquina (a versão com 24 h e
4 constelações sobrecarregou o ambiente anteriormente).

Fluxo:
1. Parâmetros de referência (leves)
2. Radiotelescópio (`uirapuru`) e orquestrador
3. Satélites GPS com PRN (refatoração NORAD ↔ Nome ↔ PRN)
4. Simulação TOD discreta (GPS)
5. Persistência HDF5/NetCDF + reload (ciclo "ida e volta")


## 1. Parâmetros de referência (leves)

Janela curta (1 h, passo 60 s) e observador do radiotelescópio `uirapuru`.
A data de referência é 1 mês antes de agora (UTC).


In [ ]:
# ── Parâmetros de referência para a simulação leve ──
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np  # noqa: F401

from radioskysims import create_radiotelescope
from radioskysims_core.domain.common import FrequencyRange, TimeRange

# Data de referência: 1 mês antes de agora (UTC).
REF_DATE = (datetime.now(UTC) - timedelta(days=30)).replace(microsecond=0)
print("Data de referência (REF_DATE):", REF_DATE.isoformat())

# Janela LEVE: 1 h a partir de REF_DATE, passo de 60 s.
_ISO = "%Y-%m-%dT%H:%M:%S"
TR = TimeRange.from_iso(
    REF_DATE.strftime(_ISO),
    (REF_DATE + timedelta(minutes=60)).strftime(_ISO),
    60,
)
print("TimeRange (1 h, 60 s):", TR.to_datetime64().shape[0], "instantes")

FR = FrequencyRange(min_mhz=1000.0, max_mhz=1250.0, resolution_mhz=0.5)

# Observador do radiotelescópio (perfil uirapuru em settings.toml).
uirapuru = create_radiotelescope("uirapuru")
OBSERVER = uirapuru.observer
print("Observador:", OBSERVER)

# Diretório de saída.
TOD_DIR = Path("data") / "TOD"
TOD_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ── Orquestrador de simulação (API desacoplada) ──
from radioskysims_simulation.application.services.sim_orchestrator import (
    SimOrchestrator,
)

orch = SimOrchestrator.from_settings(telescope="uirapuru")
print("✅ Orquestrador configurado")
print("Constelações padrão (settings):", orch.default_constellations())
print("Time range padrão (settings):", orch.default_time_range())


## 2. Satélites GPS com PRN

A refatoração integrou o mapeamento **NORAD ↔ Nome ↔ PRN** ao modelo persistido
(ver `radioskysims-sources`, branch `feature/satellite-prn-mapping`). Cada
satélite agora expõe `prn` e `sv` (ex.: `G22`), facilitando a comparação com
RINEX e com o catálogo de sinais.


In [ ]:
# ── Satélites GPS via SatelliteManager (TLE do banco) ──
from radioskysims_core.db.engine import Database
from radioskysims_sources.application.services.satellite_manager import (
    SatelliteManager,
)
from radioskysims_sources.bootstrap.builders import build_repositories

db = Database()
repos = build_repositories(db)
mgr = SatelliteManager(
    repository=repos["satellite_repository"],
    signal_repository=repos["signal_repository"],
    epoch=REF_DATE,
)

sats = mgr.create_from_constellations(["GPS"])
print(f"✅ {len(sats)} satélites GPS carregados (TLE na época de referência)")

# NORAD ↔ Nome ↔ PRN (refatoração integrada ao modelo persistido).
print(f"{'NORAD':>8} | {'Nome':<28} | {'PRN':>4} | SV")
for sat in sats[:10]:
    prn = sat.prn if sat.prn is not None else None
    sv = sat.sv if sat.prn is not None else None
    print(
        f"{sat.norad_id:>8} | {sat.name:<28} | {prn if prn is not None else '—':>4} | {sv if sv is not None else '—'}"
    )


## 3. Simulação TOD (discreta, leve)

Gera o TOD dos satélites GPS sobre o pointing do `uirapuru`, usando o
`ResourceAwareProcessor` (`use_processor=True`) para batches seguros de memória.
O dataset permanece **lazy (dask)** — não materializamos o TOD completo em RAM.


In [ ]:
# ── Simulação TOD discreta (GPS, janela de 1 h) ──
import time

t0 = time.monotonic()
result = orch.run(
    constellations=["GPS"],
    time_range=TR,
    include_satellites=True,
    include_local_sources=False,
    include_radio_sources=False,
    use_processor=True,
)
print(f"⏱ run() completou em {time.monotonic() - t0:.1f}s")

ds = result.simulation_result.dataset
print("dims:", dict(ds.sizes))
print("variáveis:", list(ds.data_vars))

# Resumo (lazy-safe — não computa o dataset completo).
print(result.format_summary()[:1200])


In [ ]:
# ── Persistência + reload (ciclo "ida e volta") ──
from radioskysims_simulation.domain.simulation import (
    SimulationOrchestratorResult,
)

out = str(TOD_DIR / "gps_tod_leve.nc")
sidecar = result.save_result(out)
print("Salvo dataset:", out)
print("Sidecar JSON :", sidecar)

# Reload sem re-rodar a simulação:
loaded = SimulationOrchestratorResult.load_result(out)
print("Reload OK. Resumo:")
print(loaded.format_summary()[:600])


## 4. Ciclo ida e volta de mapas (leve)

Fechamos o ciclo **mapas → TOD → mapas** usando os componentes do próprio
orquestrador (`orch.context`), sem recarregar a máquina: em vez da banda
completa do `uirapuru` (que no `run(include_galaxy=True)` geraria ~5001 mapas
PySM3 com resolução 0,05 MHz), usamos uma **banda reduzida de 2 canais** e
`nside=8`.

Fluxo:
1. **Ida**: mapas galácticos (PySM3) persistidos em HDF5 → projeção no
   pointing de trânsito do `uirapuru` → TOD difuso `(time, frequency)`.
2. **Volta**: `MapMakingService` (binned HEALPix) reconstrói um mapa por
   canal a partir do TOD + pointing.
3. **Validação**: comparação entre o mapa de entrada e o reconstruído nos
   pixels observados.

In [ ]:
# ── Ida: mapas galácticos (PySM3) — banda reduzida + persistência HDF5 ──
import time

from radioskysims_core.domain.common import FrequencyRange
from radioskysims_sources.domain.radiosources.continuum import Galaxy

# Banda leve: 2 canais (1000 e 1250 MHz) em vez da banda completa do uirapuru.
GAL_FR = FrequencyRange(min_mhz=1000.0, max_mhz=1250.0, resolution_mhz=250.0)
NSIDE = 8

ctx = orch.context
galaxy = Galaxy()

t0 = time.monotonic()
galaxy_mapset = ctx.galaxy_map_generator.execute(
    source=galaxy,
    frequency_range=GAL_FR,
    nside=NSIDE,
    persist=True,  # grava HDF5 em data/aux/skymaps (galaxy_<freq>mhz_nside<n>)
    overwrite=True,
)
print(
    f"✅ {len(galaxy_mapset.maps)} mapas galácticos PySM3 (nside={NSIDE}) em {time.monotonic() - t0:.1f}s"
)

for sky in galaxy_mapset.maps:
    print(
        f"  {sky.freq_ghz:.3f} GHz | {sky.unit} | frame={sky.frame} | mean={float(sky.data.mean()):.3e}"
    )

# Reload direto do repositório HDF5 (prova persistência → leitura):
loaded_galaxy = ctx.skymap_repository.load(f"galaxy_1000mhz_nside{NSIDE}")
print("Reload HDF5 OK:", loaded_galaxy is not None, "| freq_ghz:", loaded_galaxy.freq_ghz)

In [ ]:
# ── Ida: pointing de trânsito do uirapuru + projeção (mapas → TOD) ──
from radioskysims_core.domain.geometry import compute_pointing_track

# Pointing fixo alt/az do perfil uirapuru (settings.toml) → RA/Dec aparente.
time_grid = TR.to_datetime64()
pointing = compute_pointing_track(
    observer=ctx.observer,
    time_grid=time_grid,
    alt_deg=65.0,
    az_deg=0.0,
)
print(
    "Pointing track:",
    len(pointing.time),
    "instantes | ra0:",
    float(pointing["ra"].values[0]),
    "deg",
)

t0 = time.monotonic()
diffuse_tod = ctx.tod_orchestrator.run_maps(
    maps=galaxy_mapset.maps,
    pointing_track=pointing,
    projector=ctx.skymap_tod_projector,
    use_beam=True,  # convolução com o beam do uirapuru
).compute()
print(f"✅ TOD difuso (mapas → TOD) em {time.monotonic() - t0:.1f}s")
print("dims:", dict(diffuse_tod.sizes), "| vars:", list(diffuse_tod.data_vars))
print("T_antenna média por canal (K):", diffuse_tod.T_antenna.mean("time").values.round(4))

In [ ]:
# ── Volta: TOD → mapas (binned HEALPix) ──
from radioskysims_simulation.application.mapmaking import (
    GenerateSkyMapsFromTODUseCase,
)
from radioskysims_simulation.application.services.mapmaking_service import (
    MapMakingService,
)
from radioskysims_simulation.infrastructure.mapmaking import (
    HealpyBinnedMapMakerAdapter,
)

mapmaking = MapMakingService(
    generator=GenerateSkyMapsFromTODUseCase(
        mapmaker=HealpyBinnedMapMakerAdapter(),
    ),
)

t0 = time.monotonic()
mk_result = mapmaking.run(
    tod=diffuse_tod,  # TOD difuso (time, frequency) em K
    pointing_track=pointing,
    nside=NSIDE,
    frame="equatorial",
)
print(
    f"✅ Mapas reconstruídos (TOD → mapas) em {time.monotonic() - t0:.1f}s "
    f"({len(mk_result.mapmaking.results)} canais)"
)

for rec in mk_result.mapmaking.results:
    sky = rec.skymap
    print(
        f"  {sky.freq_ghz:.3f} GHz | frame={sky.frame} | n_obs={int(rec.observed_mask.sum())} "
        f"| coverage={rec.coverage_fraction:.3f} | mean_obs={float(sky.data[rec.observed_mask].mean()):.4f} K"
    )

In [ ]:
# ── Validação: mapa de entrada (galáctico) × mapa reconstruído (equatorial) ──
import healpy as hp
import numpy as np

# Mapa reconstruído do canal 1000 MHz está em coordenadas equatoriais.
rec0 = mk_result.mapmaking.results[0]
obs_pix = np.where(rec0.observed_mask)[0]

# Direções (equatoriais) dos pixels observados → coordenadas galácticas.
rot = hp.Rotator(coord=["C", "G"])
theta_eq, phi_eq = hp.pix2ang(NSIDE, obs_pix, nest=False)
theta_g, phi_g = rot(theta_eq, phi_eq)

# Amostra o mapa galáctico de ENTRADA nas mesmas direções (uK_CMB → K).
gal0 = galaxy_mapset.maps[0]
input_k = hp.get_interp_val(gal0.data * 1e-6, theta_g, phi_g)
output_k = rec0.skymap.data[obs_pix]

corr = float(np.corrcoef(input_k, output_k)[0, 1])
mae = float(np.mean(np.abs(input_k - output_k)))

print(f"Pixels observados: {len(obs_pix)}")
print(f"Correlação (entrada vs reconstruído): {corr:.3f}")
print(f"Erro médio absoluto (MAE): {mae:.4f} K")
print(
    f"\nCiclo ida e volta OK — mapas → TOD → mapas "
    f"(banda {GAL_FR.value[0]:.0f}–{GAL_FR.value[-1]:.0f} MHz, nside={NSIDE})"
)

## 5. Próximos passos (modos pesados / opcionais)

A seção 4 demonstrou o ciclo **mapas → TOD → mapas** em modo leve (banda
reduzida, `nside=8`). Os itens abaixo usam a banda completa ou fontes extras e
devem ser ligados apenas quando houver recurso:

- **Galáxia (PySM3) em banda completa**: `orch.run(include_galaxy=True, galaxy_nside=64, persist_skymaps=True)` → ~5001 mapas HEALPix no default do `uirapuru` (parte pesada; por padrão desligada).
- **NVSS**: `include_radio_sources=True` com filtro pela declinação do pointing.
- **Comparação TLE vs RINEX**: mesmo satélite nos dois tracks (`gnsstools4astro`).
- **Waterfall**: visualização do TOD com `radioskysims_waterfall`.